# Supplier Intelligence Dashboard
### Manufacturing AI/ML Series - Playbook 3 | ForgeFlow Reference Platform

**What this notebook builds:**  
A supplier intelligence dashboard that crosses spend concentration with performance grades to surface risk-adjusted exposure.

**Key findings:**
- One supplier (Purohit Steel) holds 66% of total procurement spend
- 23.6% of spend sits with C-grade or D-grade vendors
- HHI concentration score of 4,710 - well above the 2,500 "highly concentrated" threshold
- One sole-source dependency (tooling) that spend share alone misses

**No API key required.** Pure pandas + matplotlib on ForgeFlow JSON data.

**Builds on:** [AI Procurement Assistant](../ai-procurement-assistant/ai_procurement_assistant.ipynb) - supplier grades used here come from that scorecard.

---
**Data source:** [ForgeFlow Reference Platform](https://github.com/Nub-Labs/forgeflow-reference-platform) - fictional but realistic precision engineering ERP  
**Playbook:** https://nublabs.com/playbooks/manufacturing/supplier-intelligence-dashboard

## 1. Setup

In [ ]:
# Install dependencies (Colab already has these; safe to run)
!pip install pandas matplotlib --quiet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ForgeFlow raw data URL - change BASE to a local path for offline use
BASE = 'https://raw.githubusercontent.com/Nub-Labs/forgeflow-reference-platform/main/data'

print('Setup complete.')

## 2. Load data

In [ ]:
pos       = pd.read_json(f'{BASE}/procurement/purchase_orders.json')
suppliers = pd.read_json(f'{BASE}/master/suppliers.json')

print(f'Purchase orders: {len(pos)} rows')
print(f'Suppliers master: {len(suppliers)} rows')
pos.head(3)

## 3. Spend share per vendor

Group purchase orders by supplier, sum total spend for the 6-month period, calculate share.

In [ ]:
spend = (
    pos.groupby('supplier')['amount']
    .sum()
    .reset_index()
    .rename(columns={'amount': 'spend'})
    .sort_values('spend', ascending=False)
)

total_spend = spend['spend'].sum()
spend['spend_pct'] = (spend['spend'] / total_spend * 100).round(1)
spend['pos'] = pos.groupby('supplier').size().reindex(spend['supplier']).values

print(f'Total procurement spend (Jan-Jun 2026): Rs {total_spend:,.0f}')
print()
spend

## 4. Herfindahl-Hirschman Index (HHI)

HHI = sum of squared spend shares (as percentages).  
DOJ thresholds: < 1,500 unconcentrated · 1,500–2,500 moderate · > 2,500 highly concentrated

In [ ]:
hhi = (spend['spend_pct'] ** 2).sum()

if hhi < 1500:
    band = 'Unconcentrated'
elif hhi < 2500:
    band = 'Moderately concentrated'
else:
    band = 'HIGHLY CONCENTRATED'

print(f'HHI = {hhi:.0f}  →  {band}')
print()
print('Breakdown:')
for _, row in spend.iterrows():
    contrib = row['spend_pct'] ** 2
    print(f"  {row['supplier'][:35]:<35} {row['spend_pct']:5.1f}%  squared → {contrib:7.1f}")

## 5. Overlay supplier grades (from AI Procurement Assistant scorecard)

These grades come from the composite scorecard in Playbook 1.  
In production you would join from a database table; here we replicate the output.

In [ ]:
# Scorecard output from Playbook 1 (AI Procurement Assistant)
scorecard = pd.DataFrame([
    {'supplier': 'Purohit Steel Alloys Pvt Ltd',   'short': 'Purohit',      'grade': 'A', 'score': 94, 'sole_source': False, 'category': 'Steel - EN8'},
    {'supplier': 'Sudarshan Metal Trading Co',      'short': 'Sudarshan',    'grade': 'C', 'score': 51, 'sole_source': False, 'category': 'Steel - EN24'},
    {'supplier': 'Paschim Steel Corporation',       'short': 'Paschim',      'grade': 'D', 'score': 28, 'sole_source': False, 'category': 'Stainless / Aluminium'},
    {'supplier': 'Amrit Lubricants Pvt Ltd',        'short': 'Amrit',        'grade': 'A', 'score': 95, 'sole_source': False, 'category': 'Consumables'},
    {'supplier': 'Vishwakarma Tooling Pvt Ltd',     'short': 'Vishwakarma',  'grade': 'A', 'score': 83, 'sole_source': True,  'category': 'Tooling'},
])

# Join spend with grades
dashboard = spend.merge(scorecard, on='supplier', how='left')
dashboard['risk_label'] = dashboard['grade'].map({'A': 'Low risk', 'B': 'Medium risk', 'C': 'Medium risk', 'D': 'High risk'})

print('Supplier Intelligence Dashboard:')
print(dashboard[['short', 'grade', 'score', 'spend', 'spend_pct', 'risk_label', 'sole_source', 'category']].to_string(index=False))

## 6. Risk-adjusted spend exposure

In [ ]:
at_risk = dashboard[dashboard['grade'].isin(['C', 'D'])]
at_risk_spend = at_risk['spend'].sum()
at_risk_pct = at_risk['spend'].sum() / total_spend * 100

sole_source = dashboard[dashboard['sole_source'] == True]

print('=== Risk-Adjusted Spend Summary ===')
print(f'Total procurement spend:   Rs {total_spend:>12,.0f}')
print(f'Low-risk spend (A-grade):  Rs {dashboard[dashboard["grade"]=="A"]["spend"].sum():>12,.0f}  ({dashboard[dashboard["grade"]=="A"]["spend_pct"].sum():.1f}%)')
print(f'Medium-risk spend (C):     Rs {dashboard[dashboard["grade"]=="C"]["spend"].sum():>12,.0f}  ({dashboard[dashboard["grade"]=="C"]["spend_pct"].sum():.1f}%)')
print(f'High-risk spend (D):       Rs {dashboard[dashboard["grade"]=="D"]["spend"].sum():>12,.0f}  ({dashboard[dashboard["grade"]=="D"]["spend_pct"].sum():.1f}%)')
print(f'TOTAL AT-RISK (C+D):       Rs {at_risk_spend:>12,.0f}  ({at_risk_pct:.1f}%)')
print()
print('=== At-Risk Vendors ===')
for _, row in at_risk.iterrows():
    print(f"  {row['short']}: Grade {row['grade']} · Rs {row['spend']:,.0f} ({row['spend_pct']}%) · {row['category']}")
print()
print('=== Sole-Source Dependencies ===')
for _, row in sole_source.iterrows():
    print(f"  {row['short']}: Grade {row['grade']} · {row['spend_pct']}% of spend · No alternative qualified")

## 7. Visualisation - spend concentration with risk overlay

In [ ]:
grade_colors = {'A': '#10b981', 'B': '#3b82f6', 'C': '#f59e0b', 'D': '#ef4444'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ForgeFlow Supplier Intelligence Dashboard - Jan to Jun 2026', fontsize=13, fontweight='bold', y=1.02)

# --- Left: spend bar chart with grade colour ---
ax = axes[0]
colors = [grade_colors[g] for g in dashboard['grade']]
bars = ax.barh(dashboard['short'], dashboard['spend_pct'], color=colors, edgecolor='white', linewidth=0.5)

for bar, row in zip(bars, dashboard.itertuples()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{row.spend_pct}%  ({row.grade})', va='center', fontsize=9)

ax.set_xlabel('% of total procurement spend')
ax.set_title('Spend share by supplier\n(colour = performance grade)', fontsize=10)
ax.set_xlim(0, 85)
ax.invert_yaxis()
ax.axvline(x=50, color='#94a3b8', linestyle='--', linewidth=0.8, label='50% single-vendor threshold')
ax.legend(fontsize=8)

legend_patches = [mpatches.Patch(color=c, label=f'Grade {g}') for g, c in grade_colors.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8)

# --- Right: risk-adjusted spend donut ---
ax2 = axes[1]
risk_groups = dashboard.groupby('risk_label')['spend'].sum()
risk_colors = {'Low risk': '#10b981', 'Medium risk': '#f59e0b', 'High risk': '#ef4444'}
labels = [f"{label}\nRs {val/100000:.1f}L" for label, val in risk_groups.items()]
wedge_colors = [risk_colors.get(l, '#94a3b8') for l in risk_groups.index]

wedges, texts, autotexts = ax2.pie(
    risk_groups.values,
    labels=labels,
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.55),
    textprops={'fontsize': 9}
)
ax2.set_title('Risk-adjusted spend exposure\n(C-grade + D-grade = at risk)', fontsize=10)
ax2.text(0, 0, f'HHI\n{hhi:.0f}', ha='center', va='center', fontsize=11, fontweight='bold', color='#1e293b')

plt.tight_layout()
plt.savefig('supplier_intelligence_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as supplier_intelligence_dashboard.png')

## 8. Month-by-month spend trend per vendor

In [ ]:
pos['order_month'] = pd.to_datetime(pos['order_date']).dt.to_period('M')
monthly = pos.groupby(['order_month', 'supplier'])['amount'].sum().reset_index()
monthly['short'] = monthly['supplier'].map(dict(zip(scorecard['supplier'], scorecard['short'])))

pivot = monthly.pivot(index='order_month', columns='short', values='amount').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
for col in pivot.columns:
    grade = scorecard.set_index('short').loc[col, 'grade'] if col in scorecard['short'].values else 'A'
    ax.plot(pivot.index.astype(str), pivot[col], marker='o', label=col, color=grade_colors.get(grade, '#94a3b8'), linewidth=2)

ax.set_title('Monthly spend per supplier - Jan to Jun 2026', fontsize=11)
ax.set_xlabel('Month')
ax.set_ylabel('Spend (Rs)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rs {x/100000:.1f}L'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Recommendations

In [ ]:
print('=== SUPPLIER INTELLIGENCE RECOMMENDATIONS ===')
print()

for _, row in dashboard.sort_values('spend', ascending=False).iterrows():
    print(f"[{row['grade']}] {row['short']} - Rs {row['spend']/100000:.1f}L ({row['spend_pct']}%)")

    if row['grade'] == 'D':
        print(f"  ACTION: STOP all new POs immediately. Source {row['category']} from alternative before next procurement cycle.")
    elif row['grade'] == 'C':
        print(f"  ACTION: Do not increase share. Review two clean quarters before expanding volume.")
    elif row['spend_pct'] > 50:
        print(f"  ACTION: Qualify a second {row['category']} vendor. Target ≤40% share for this supplier.")
    elif row['sole_source']:
        print(f"  ACTION: Begin qualification of alternative {row['category']} supplier. Document sole-source risk in risk register.")
    else:
        print(f"  STATUS: No action required.")
    print()

print(f'HHI: {hhi:.0f} - Highly concentrated (target: below 2,500 through vendor diversification)')
print(f'At-risk spend: Rs {at_risk_spend/100000:.1f}L ({at_risk_pct:.1f}%) - reduce below 10% by moving Paschim volume')

---
## What's next

This playbook completes the supply-side picture:
- **Playbook 1** scored suppliers by quality, delivery, and lead time
- **Playbook 2** calibrated production lead times and identified the Heavy Press bottleneck
- **Playbook 3** (this notebook) crossed spend with grades to surface financial risk exposure

**Playbook 4 - Production Workflow Agent** moves from analysis to action:  
Build an agent that reads job card state in real time, detects overdue work orders, flags the bottleneck workstation, and generates a daily briefing - with an optional LLM extension for natural language summarisation.

→ [production_workflow_agent.ipynb](../production-workflow-agent/production_workflow_agent.ipynb)